<div style="margin: 0 auto 30px; height: 100px; border: 2px solid gray; border-radius: 6px;">
  <div style="float: left; margin: 5px 10px 5px 10px; "><img src="img/bfh.jpg" /></div>
  <div style="float: right; margin: 20px 30px 0; font-size: 15pt; font-weight: bold; color: #98b7d2;"><a href="https://moodle.bfh.ch/course/view.php?id=39255" style="color: #98b7d2;">BTE5476 - Project-Oriented Digital Signal Processing </a></div>
</div>
<div style="clear: both; margin: 20px 0; font-size: 30pt; font-weight: bold; color: #64788b; margin-left: 30px;">
    Digital Biquads (with Audio Applications)
</div>

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio 
from scipy import signal

import warnings
import importnb
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with importnb.Notebook(include_markdown_docstring=False): 
        import FilterUtils as fu

In [ ]:
plt.rcParams['figure.figsize'] = 14, 4 
matplotlib.rcParams.update({'font.size': 14})

In [ ]:
def draw_lines(handle, x, y):
    for c in x:
        handle.axvline(x=c, linewidth=0.5, color='r');
    for c in y:
        handle.axhline(y=c, linewidth=0.5, color='r'); 

In [ ]:
def play(x, sf):
    TAPER = 1000
    y = np.copy(x)
    y[:TAPER] *= np.linspace(0, 1, TAPER)
    y[-TAPER:] *= np.linspace(1, 0, TAPER)
    return Audio(y, rate=sf)

import ipywidgets as widgets
def multiplay(clips, sf, title=None):
    outs = [widgets.Output() for c in clips]
    for ix, item in enumerate(clips):
        with outs[ix]:
            display(play(item, sf))
    return widgets.HBox(outs)

In [ ]:
DEFAULT_SF = 16000

# Audio signal processing: the world of log-log

Let's look at the simplest IIR lowpass, the leaky integrator:

$$
  y[n] = \lambda y[n-1] + (1-\lambda) x[n]
$$

In signal processing, we usually display the magnitude response using linear units like so:

In [ ]:
lam = 0.9
fu.filter_props([1 - lam], [1, -lam])

In audio applications, the same magnitude response would be displayed as a log-log plot like so:

In [ ]:
fu.analog_response([1 - lam], [1, -lam], DEFAULT_SF, dB=-50)

NB: these plotting functions are available in the companion notebook ``FilterUtils.ipynb``

## The frequency axis
 
 * analog filter specifications use frequency values in Hz instead of normalized values over $[-\pi, \pi]$
 * the analog frequency axis is usually expressed in logarithmic units (correlates better with audio perception, energy emissions, etc)
   * with $\log_{10}$ the frequency axis is labeled in **decades**
   * with $\log_2$ the frequency axis is labeled in **octaves**
 
Analog specs must be mapped to digital specs via the sampling frequency of the final DSP application: 

**the filter coefficients depend on the sampling frequency**

## Logarithmic magnitude

 * magnitude response usually displayed in dB (i.e. on a log scale)
    * _attenuation_ is measured as the amplitude (or power) _ratio_ between input and output
    * in a cascade of filters, log attenuations are additive
    



### Refresher: the decibel (dB)

Historically, a logarithmic measure of **power loss** over telecommunication cables. 

One dB was defined as the attenuation incurred by an electric signal over 1 mile of copper cable.

dB for **power ratios**: always relative to a _reference_ power $P_0$:

$$
    G = 10\log_{10}\frac{P}{P_0} \quad\mathrm{dB}
$$

 * positive for gain, negative for loss
 * +3 dB = twice the power, -3 dB = half the power
 * +10 dB = ten times the power
 
 * To go back to linear units: $P = P_0 \cdot 10^{G/10}$ (Watts)


In most cases power is proportional to amplitude squared; for example:
 * $P = V^2/R$ in electrical circuits
 * $E = mv^2/2$ for the kinetic energy
 * $I = p^2/Z$ for sound intensity. 

dB for **amplitude ratios**: always relative to a _reference_ amplitude $V_0$:

$$
    G= 10\log_{10}\frac{P}{P_0} = 10\log_{10}\frac{V^2}{V_0^2} = 20\log_{10}\frac{V}{V_0} \quad\mathrm{dB}
$$

 * +6 dB = twice the amplitude, -6 dB = half the amplitude
 * +20 dB = ten times the amplitude
 
 * To go back to linear units $V = V_0 \cdot 10^{G/20}$ (Volts)

**Specialized units:**

  * **dBm** : power ratio with reference $P_0 = 1~\mathrm{mW}$
  * **dBu** : voltage (RMS) ratio with reference $v_0 = 0.7746~\mathrm{V}$ ($v_0$ is the voltage drop across a $600~\Omega$ resistor dissipating $1~\mathrm{mW}$; so, numerically, dBu = dBm)
  * **dBV** : voltage (RMS) ratio with reference $v_0 = 1~\mathrm{V}$ (dBV = dBu + 2.2)
  * **dBFS**: _peak_ amplitude ratio for digital signals at $R$ bps with reference $v_0 = 2^R$; always negative.
  

**Example: Mic and Line audio levels:**

 * mic level: between -60 and -40 dBu (~1 mV RMS)
 * line level (consumer): 4 dBu (~1 V RMS) 
 * line level (pro): -10 dBV (-7.8 dBu, ~0.3 V RMS)

    
<img src="img/spl.png" alt="rlc" style="float: right; width: 700px; margin: 40px;"/>
    
**The human hearing system:**

  * perceived loudness is proportional to log of power
  * ears have an enormous dynamic range: 130 dB or $1:10^{13}$ 
     
 

# The versatile second-order section (aka, the biquad)

<img src="img/biquad.png" alt="Drawing" style="float: right; width: 600px; margin: 00px 30px;"/>

<br>

$$
    H(z) = \frac{b_0 + b_1 z^{-1} + b_{2}z^{-2}}{1 + a_1 z^{-1} + a_{2}z^{-2}}
$$

<br>

 * if $a_2 = b_2 = 0$ we get a first-order section
 * if $a_1 = a_2 = 0$ we get a two-tap FIR


## Biquads as standalone filters

Biquads are quite good on their own and this notebook is baiscally a _cookbook_  with recipes to build second-order filters:
 * lowpass, highpass, bandpass
 * notches and resonators
 * shelves
 * equalizers
 
 Most of these filters are used in audio applications, and we will use audio examples to test our design.



The recipes have been adapted from Robert Bristow-Johnson's famous [cookbook](https://webaudio.github.io/Audio-EQ-Cookbook/audio-eq-cookbook.html). 

Each function returns the five coefficients in the second-order transfer function

$$
    H(z) = \frac{b_0 + b_1 z^{-1} + b_{2}z^{-2}}{1 + a_1 z^{-1} + a_{2}z^{-2}} 
$$

## The biquad as a building block

 * every transfer function can be expressed as a cascade of second-order sections:
 $$
    \frac{b_0 + b_1 z^{-1} + \ldots + b_{N-1}z^{-N+1}}{1 + a_1 z^{-1} + \ldots + a_{N-1}z^{-N+1}} = \prod_{k=0}^{\lceil N/2 \rceil} \frac{b_{k,0} + b_{k,1} z^{-1} + b_{k,2}z^{-2}}{1 + a_{k,1} z^{-1} + a_{k,2}z^{-2}} $$
 * the value of the poles is a complicated function of the filter coefficients
 * coefficients are quantized $\Rightarrow$ poles can drift out of unit circle
 * in cascade form the positions of the poles are easier to check/control

## The biquad's stability

<img src="img/stability_triangle.png" alt="Drawing" style="float: right; width: 600px; margin: 0px 30px;"/>

<br><br>

$$
    H(z) = \frac{b_0 + b_1 z^{-1} + b_{2}z^{-2}}{1 + a_1 z^{-1} + a_{2}z^{-2}}
$$

<br>

 * optimized biquad implementations for different types of precision (fixed point, floating point, etc)

## Factorization of high-order filters

Reliable algorithms exist to design high-order filters (Butterworth, Chebyshev, elliptic, etc.) but direct implementation of high-order CCDEs is prone to problems:

 * coefficient quantization causes pole drift
 * numerical instability for "high performance" filters (i.e. steep transition bands, small bandwidth, etc)
 
In this example, note how reducing the precision of the coefficients to 4 decimal digits causes the filter to completely miss the specs:

In [ ]:
CUTOFF = 1000
ELLIPTIC_PARAMS = [6, 1, 40, 1000]
eb, ea = signal.ellip(*ELLIPTIC_PARAMS, fs=DEFAULT_SF)

PRECISION = 4   # decimal digits

_, ax = plt.subplots()
fu.analog_response(eb, ea, DEFAULT_SF, dB=-80, axis=ax, label='6th order elliptic')
fu.analog_response(np.round(eb, PRECISION), np.round(ea, PRECISION), DEFAULT_SF, dB=-80, axis=ax, color=f'C3:', label='6th order elliptic, quantized')
draw_lines(plt, [CUTOFF], [-3])
ax.legend(loc='lower left');

An efficient solution is to factor the transfer function into second order sections and implement the filter as a cascade of biquads:

$$
    H(z) = \frac{b_0 + b_1 z^{-1} + \ldots + b_{N-1}z^{-N+1}}{1 + a_1 z^{-1} + \ldots + a_{N-1}z^{-N+1}} = \prod_{k=0}^{N/2} \frac{b_{k,0} + b_{k,1} z^{-1} + b_{k,2}z^{-2}}{1 + a_{k,1} z^{-1} + a_{k,2}z^{-2}}
$$

Luckily, we can easily obtain a filter in cascaded form by specifying `output=sos` in the Scipy design functions.

In the following example, the second-order sections are quantized individually and the cascade performs very closely to the non-quantized filter:

In [ ]:
_, ax = plt.subplots()
fu.analog_response(eb, ea, DEFAULT_SF, dB=-80, axis=ax, label='6th order elliptic')

# this returns an array of second-order filter coefficients. Each row corresponds to a section, 
#  with the first three columns providing the numerator coefficients and the last three providing the denominator
soe = signal.ellip(*ELLIPTIC_PARAMS, fs=DEFAULT_SF, output='sos')
cb, ca = [1], [1]
# build the high-order filter by cascading the quantized second-order sections
for n in range(0, 3):
    b, a = np.round(soe[n][0:3], PRECISION), np.round(soe[n][3:6], PRECISION)
    fu.analog_response(b, a, DEFAULT_SF, dB=-80, axis=ax, color=f'C{n}:', label=f'section {n+1}, quantized')
    cb = np.polymul(b, cb)
    ca = np.polymul(a, ca)
fu.analog_response(cb, ca, DEFAULT_SF, dB=-80, axis=ax, color='C3:', label='cascade of quantized sections')
draw_lines(plt, [CUTOFF], [-3])
ax.legend(loc='lower left');

# The biquad cookbook

(The following recipes are also available as a stripped-down list in the notebook "BiquadRecipes.ipynb")

## Lowpass

Specs:
 1. quality factor $Q$; default is $Q = \sqrt{1/2}$ for a monotonic, maximally flat characteristic (Butterworth)
 1. _corner frequency_ $f_c$ (aka _cutoff_ ); $|H(f_c)| = Q$ (for $Q = \sqrt{1/2}$, attenuation at $f_c$ is $-3$ dB)

In [ ]:
def LPF(fc, sf, Q=(1/np.sqrt(2))):
    """Biquad lowpass filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.sin(w) / (2 * Q)
    c = np.cos(w)
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    b = np.array([(1 - c) / 2, 1 - c, (1 - c) / 2])
    return b / a[0], a / a[0]

In [ ]:
CUTOFF = 1000
b, a = LPF(CUTOFF, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-50)
draw_lines(plt, [CUTOFF], [-3])

In [ ]:
fu.filter_props(b, a)
draw_lines(plt.gcf().get_axes()[0], [2 * np.pi * CUTOFF / DEFAULT_SF], [np.sqrt(0.5)])

### Resonance peak

If $Q > 1/\sqrt{2}$, the magnitude response has a **peak** at $f_c$.

In the time domain, this corresponds to a damped oscillatory impulse response.

These $Q$ values become useful when cascading multiple sections, as we will see later.

In [ ]:
_, (fr, ir) = plt.subplots(2, figsize=(16,9))
CUTOFF = 100
for n, q in enumerate([0.1, 0.5, 1/np.sqrt(2), 5, 20]):
    b, a = LPF(CUTOFF, DEFAULT_SF, Q=q)
    fu.analog_response(b, a, DEFAULT_SF, dB=-50, axis=fr, color=f'C{n}', label=f'Q={q:.2f}')
    ir.plot(signal.lfilter(b, a, np.r_[1, np.zeros(2000)]))
draw_lines(fr, [CUTOFF], [-3])
fr.legend();

## Highpass

A highpass filter is simply the complementary filter to a lowpass, with the same roles for $f_c$ and $Q$.

In [ ]:
def HPF(fc, sf, Q=(1/np.sqrt(2))):
    """Biquad highpass filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.sin(w) / (2 * Q)
    c = np.cos(w)
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    b = np.array([(1 + c) / 2, -1 - c, (1 + c) / 2])
    return b / a[0], a / a[0]

In [ ]:
CUTOFF = 2500
b, a = HPF(CUTOFF, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-50)
draw_lines(plt, [CUTOFF], [-3])

In [ ]:
fu.filter_props(b, a)
draw_lines(plt.gcf().get_axes()[0], [2 * np.pi * CUTOFF / DEFAULT_SF], [np.sqrt(0.5)])

## Cascades of lowpass and highpass biquads

Often the performance of a second-order filter is not enough.

 * higher-order filters can be decomposed into second-order stages
 * "canonical" biquads can be repeated to improve or change overall performance

Applications:
 * increase rejection in stopband
 * combine shelves to provide tone control
 * design multiband equalizers

 * biquad's transfer function $H(z)$
 * cascade $N$ times
 * overall transfer function $H_c(z) = H^N(z)$
 * magnitude response in dB multiplied by $N$

In [ ]:
def iterated_filter(sections, fc, sf, recipe=LPF):
    cb, ca = [1], [1]
    for n in range(0, sections):
        b, a = recipe(fc, sf)
        ca = np.polymul(a, ca)
        cb = np.polymul(b, cb)   
    return cb, ca

Example: cascade of five second-order Butterworth lowpass

In [ ]:
_, ax = plt.subplots()
CUTOFF = 1000
b, a = LPF(CUTOFF, DEFAULT_SF)
cb, ca = b, a
for n in range(0, 5):
    fu.analog_response(cb, ca, DEFAULT_SF, dB=-60, axis=ax, color=f'C{n}', label=f'{n+1} sections')
    ca = np.polymul(a, ca)
    cb = np.polymul(b, cb)
plt.axvline(x=CUTOFF, linewidth=0.5, color='r')    
plt.axhline(y=-3, linewidth=0.5, color='r')    
plt.axhline(y=-15, linewidth=0.5, color='r');    
ax.legend(loc='lower left');

Good things:
  * still monotonic
  * steeper transition to stopband

However:
  * cutoff frequency moves
  * bandwidth decreases
  
It's better to use Python and get a high-order filtered in factored form, but for a quick prototype a simple iterated cascade can be OK

## Bandpass

Approximately unit gain (0 dB) in the passband. Specs:

 1. center frequency $f_c$
 1. bandwidth $b = (f_+ - f_-)$, frequencies where attenuation reaches $-3$ dB. Passband is almost but not exactly symmetric around $f_c$ (because of bilinear transform)

In [ ]:
def BPF(fc, bw, sf):
    """Biquad bandpass filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.tan(np. pi * bw / sf)
    c = np.cos(w)
    b = np.array([alpha, 0, -alpha])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]

In [ ]:
CENTER, BANDWIDTH = 1000, 400
b, a = BPF(CENTER, BANDWIDTH, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-40)
draw_lines(plt, [CENTER - BANDWIDTH / 2, CENTER, CENTER + BANDWIDTH / 2], [-3])

In [ ]:
fu.filter_props(b, a)
draw_lines(plt, [2 * np.pi * CENTER / DEFAULT_SF], [np.sqrt(0.5)])

## Resonator

When the bandwith is very small, the second order bandpass becomes a constant-gain resonator:

In [ ]:
_, ax = plt.subplots()
BANDWIDTH = 10
FC = [100, 1000, 2000, 4000, 6000]
for n, fc in enumerate(FC):
    b, a = BPF(fc, BANDWIDTH, DEFAULT_SF)
    fu.frequency_response(b, a, dB=-50, half=True, axis=ax);

## Notch

Complementary filter to a resonator:
 * infinite attenuation at $f_c$ (zeros on unit circle)
 * very small bandwidth (poles very close to zeros)

In [ ]:
def notch(fc, bw, sf):
    """Biquad notch filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.tan(np. pi * bw / sf)
    c = np.cos(w)
    b = np.array([1, -2 * c, 1])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]

In [ ]:
CENTER, BANDWIDTH = 2000, 100
b, a = notch(CENTER, BANDWIDTH, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-40)
draw_lines(plt, [CENTER], [-3])

In [ ]:
fu.filter_props(b, a)
draw_lines(plt.gcf().get_axes()[0], [2 * np.pi * CENTER / DEFAULT_SF], [np.sqrt(0.5)])

## Shelves

Shelving filters are used to amplify either the low or the high end of a signal's spectrum; common in consumer audio appliances, behind the standard "Bass" and "Treble" tone knobs. Specs:

 1.  _shelf gain_ in dB
 1. the midpoint frequency $f_c$, where gain reaches half its value
 1. "quality factor" $Q$, as in lowpass filters

In [ ]:
def LSH(fc, gain, sf,  Q=(1/np.sqrt(2))):
    """Biquad low shelf"""
    w = 2 * np.pi * fc / sf
    A = 10 ** (gain / 40)    
    alpha = np.sin(w) / (2 * Q)
    c = np.cos(w)
    b = np.array([A * ((A + 1) - (A - 1) * c + 2 * np.sqrt(A) * alpha),
                  2 * A * ((A - 1) - (A + 1) * c),
                  A * ((A + 1) - (A - 1) * c - 2 * np.sqrt(A) * alpha)])
    a = np.array([(A + 1) + (A - 1) * c + 2 * np.sqrt(A) * alpha,
                 -2 * ((A - 1) + (A + 1) * c),
                 (A + 1) + (A - 1) * c - 2 * np.sqrt(A) * alpha])
    return b / a[0], a / a[0]

In [ ]:
MIDPOINT, GAIN_DB = 200, 40
b, a = LSH(MIDPOINT, GAIN_DB, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-40)
draw_lines(plt, [MIDPOINT], [GAIN_DB / 2])

In [ ]:
fu.filter_props(b, a);

In [ ]:
def HSH(fc, gain, sf,  Q=(1/np.sqrt(2))):
    """Biquad high shelf"""
    w = 2 * np.pi * fc / sf
    A = 10 ** (gain / 40)    
    alpha = np.sin(w) / (2 * Q)
    c = np.cos(w)
    b = np.array([A * ((A + 1) + (A - 1) * c + 2 * np.sqrt(A) * alpha),
                  -2 * A * ((A - 1) + (A + 1) * c),
                  A * ((A + 1) + (A - 1) * c - 2 * np.sqrt(A) * alpha)])
    a = np.array([(A + 1) - (A - 1) * c + 2 * np.sqrt(A) * alpha,
                 2 * ((A - 1) - (A + 1) * c),
                 (A + 1) - (A - 1) * c - 2 * np.sqrt(A) * alpha])
    return b / a[0], a / a[0]

In [ ]:
MIDPOINT, GAIN_DB = 2000, 40
b, a = HSH(MIDPOINT, GAIN_DB, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-40)
draw_lines(plt, [MIDPOINT], [GAIN_DB / 2])

In [ ]:
fu.filter_props(b, a);

### Combining shelving filters

Shelving filters may be combined to create filters to boost or attenuate a particular frequency range

In [ ]:
cb, ca = LSH(100, 20, DEFAULT_SF)    
b, a = HSH(2000, 20, DEFAULT_SF)
cb = np.polymul(b, cb) 
ca = np.polymul(a, ca)
fu.analog_response(cb, ca, DEFAULT_SF, dB=-50, points=10001);

Note that amplitudes sum up on a log scale so careful when low and high shelves intersect

In [ ]:
_, ax = plt.subplots()
GAIN = 20
bl, al = LSH(1000, GAIN, DEFAULT_SF)   
bh, ah = HSH(10, GAIN, DEFAULT_SF)
cb = np.polymul(bh, bl) 
ca = np.polymul(ah, al)
fu.analog_response(cb, ca, DEFAULT_SF, dB=-50, axis=ax, points=10001, label='cascade');
fu.analog_response(bl, al, DEFAULT_SF, dB=-50, axis=ax, points=10001, color=f'C1:', label='low shelf')
fu.analog_response(bh, ah, DEFAULT_SF, dB=-50, axis=ax, points=10001, color=f'C2:', label='high shelf')
ax.legend();

we must rescale the response to have unit gain at band edges

In [ ]:
cb *= 10 ** (-GAIN / 20)
fu.analog_response(cb, ca, DEFAULT_SF, dB=-50, points=10001);

## Peaking EQ

Fundamental ingredient in multiband equalization: a bandpass filter with user-definable gain in passband and unit gain elsewhere. Specs:

 1. desired peak gain in dB (can be negative)
 1. peak frequency $f_c$, where gain is attained
 1. bandwidth: interval around $f_c$ where gain is more than half of final gain (in dB); as in the standard bandpass, bandwdidth is not perfectly symmetrical around $f_c$ 

In [ ]:
def PEQ(fc, bw, gain, sf):
    """Peaking EQ filter """
    w = 2 * np.pi * fc / sf
    A = 10 ** (gain / 40)    
    alpha = np.tan(np. pi * bw / sf)
    c = np.cos(w)
    b = np.array([1 + alpha * A, -2 * c, 1 - alpha * A])
    a = np.array([1 + alpha / A, -2 * c, 1 - alpha / A])
    return b / a[0], a / a[0]

In [ ]:
CENTER, BW, GAIN_DB = 800, 400, 30
b, a = PEQ(CENTER, BW, GAIN_DB, DEFAULT_SF)
fu.analog_response(b, a, DEFAULT_SF, dB=-40)
draw_lines(plt, [CENTER], [GAIN_DB / 2])

In [ ]:
fu.filter_props(b, a);

Note that peaking EQ filters with opposite gains are perfectly complementary: 

In [ ]:
CENTER, BW, GAIN_DB = 800, 400, 40
b, a = PEQ(CENTER, BW, GAIN_DB, DEFAULT_SF)
y = signal.lfilter(b, a, np.r_[1, np.zeros(200)])
plt.plot(y, label='PEQ impulse response (gain > 0)')
b, a = PEQ(CENTER, BW, -GAIN_DB, DEFAULT_SF)
y = signal.lfilter(b, a, y)
plt.plot(y, label='inverse-filtered PEQ IR');
plt.legend(loc='lower right');

### Parametric equalization

Peaking equalizers with distinct bandwidths can be cascaded to obtain an arbitrary equalization curve for the entire range of input frequencies; indeed, this is the technique behind so-called _parametric equalizers_ where a bank of logarithmically spaced peaking eq's with independent gain controls allow the user to easily define a global equalization response.

In [ ]:
cb, ca = np.ones(1), np.ones(1)
for n, g in enumerate([20, -10, 40]):
    b, a = PEQ(10 ** (n+1), 10 ** (n + 1), g, DEFAULT_SF)
    cb = np.polymul(b, cb)
    ca = np.polymul(a, ca)
fu.analog_response(cb, ca, DEFAULT_SF, dB=-50, points=10001);

# Examples

In [ ]:
from IPython.display import Audio
from scipy.io import wavfile

In [ ]:
sf, groove = wavfile.read("snd/groove.wav")
groove = (groove - np.mean(groove)) / 32767.0
Audio(groove, rate=sf)

## Music over the phone

Bandwidth of phone lines is from 180 Hz to 3.2 kHz. Rolloff is sharp after upper cutoff.

A single passband biquad is not selective enough, so let's combine highpas and lowpass filters

In [ ]:
LOW, HIGH = 200, 3200
bl, al = iterated_filter(5, HIGH, sf, recipe=LPF) # use multiple sections for a steep rolloff
bh, ah = iterated_filter(2, LOW, sf, recipe=HPF)  # if we try a higher-order highpass we have stability issues
a, b = np.polymul(ah, al), np.polymul(bh, bl)
fu.analog_response(b, a, sf, dB=-40)
draw_lines(plt, [LOW, HIGH], [-3])

In [ ]:
multiplay([groove, signal.lfilter(b, a, groove)], sf)

## Bass and Treble controls

Simple tone controls on consumer appliances are based on shelf filters:
 * the bass knob controls the gain of a low shelf with cutoff around 250 Hz
 * the treble knob controls the gain of a high shelf with cutoff around 4 kHz

In [ ]:
def tone_control(bass, treble, audio, sf):
    BASS_FC, TREBLE_FC = 250, 4000
    bh, ah = HSH(TREBLE_FC, treble, sf)
    bl, al = LSH(BASS_FC, bass, sf)
    a, b = np.polymul(ah, al), np.polymul(bh, bl)
    return signal.lfilter(b, a, audio)

In [ ]:
multiplay([groove, tone_control(15, 0, groove, sf)], sf)

In [ ]:
multiplay([groove, tone_control(0, 10, groove, sf)], sf)

In [ ]:
multiplay([groove, tone_control(8, 10, groove, sf)], sf)